In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-f4d43d7f-0232-8a3e-4022-2b7e0dcfe455)


In [3]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [4]:
def iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index0, coordinate_list):

    data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index0[test_ind], coordinate_list[test_ind])

    test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

    return test_iter

In [5]:
def iter_subject_gen(train_ind, valid_ind, test_ind, test_subject_ind, brain_signal_lfp, brain_region_index0, coordinate_list):

    test_ind_od = test_subject_ind

    data_test = TensorDataset(brain_signal_lfp[test_ind_od,:], brain_region_index0[test_ind_od], coordinate_list[test_ind_od])

    test_iter_od = DataLoader(data_test, batch_size=128, shuffle=True)

    return test_iter_od

In [6]:
# @title Load data
key = 'stimOn_times'
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
# torch.save(brain_signal_lfp, '/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt')
brain_signal_lfp = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt')
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)
# list_dict_acronym_selec = torch.load(file_dir + '/list_dict_acronym_selec.pt')
brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']


In [7]:
Mat_dict = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/Confusion_mat_stimOn_times.pt', weights_only=False)

In [8]:
!pip install networkx
import networkx as nx

In [9]:
import copy

def modularity_separation(M, resolution, sort=True):
    G = nx.from_numpy_array(M)
    c = nx.community.greedy_modularity_communities(G, weight='weight', resolution=resolution)
    acronym_label = []
    community_label = []
    index = []
    for c_ii, c0 in enumerate(c):
        for c_index in c0:
            community_label.append(c_ii)
            acronym_label.append(acronym_list[c_index])
            index.append(c_index)

    community_label = np.array(community_label)
    acronym_label = np.array(acronym_label)
    index = np.array(index)

    if sort == True:
        sort_index = np.argsort(index)
        community_label = community_label[sort_index]
        acronym_label = acronym_label[sort_index]
        index = index[sort_index]

    return community_label, acronym_label, index

In [10]:
resolution = 1.0
community_number = []
community_index = {}
for name in ['AnyNet', 'ViT', 'RNN']:
    X = copy.deepcopy(Mat_dict[name])
    community_label, acronym_label, index = modularity_separation(X, resolution)
    community_number.append(len(np.unique(community_label)))
    community_index[name] = community_label

communities = []
communities_label = []
communities_acronym = []
ii = 0
for community_ii_AnyNet in range(0, community_number[0]):
    for community_ii_ViT in range(0, community_number[1]):
        for community_ii_RNN in range(0, community_number[2]):
            community0 = np.intersect1d(np.intersect1d(np.argwhere(community_index['AnyNet'] == community_ii_AnyNet).flatten(),
                                            np.argwhere(community_index['ViT'] == community_ii_ViT).flatten()),
                            np.argwhere(community_index['RNN'] == community_ii_RNN).flatten())
            if len(community0) > 2:
                communities.append(community0)
                communities_acronym.append(np.array(acronym_list)[community0])
                communities_label.append(ii * np.ones_like(community0))
                ii = ii + 1

In [11]:
communities = np.concat(communities)
communities_label = np.concat(communities_label)
communities_acronym = np.concat(communities_acronym)

In [14]:
community_dict = {}
community_dict[1.0] = {}
community_dict[resolution]['communities'] = communities
community_dict[resolution]['communities_label'] = communities_label
community_dict[resolution]['communities_acronym'] = communities_acronym

In [15]:
resolution

1.0

In [13]:
communities_label

array([ 0,  0,  0,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
        2,  2,  2,  2,  2,  2,  3,  3,  3,  3,  3,  3,  3,  3,  4,  4,  4,
        4,  4,  4,  4,  5,  5,  5,  5,  5,  6,  6,  6,  6,  7,  7,  7,  7,
        7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,
        7,  7,  7,  7,  8,  8,  8,  8,  8,  8,  8,  9,  9,  9,  9,  9,  9,
        9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,
        9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,
        9,  9,  9,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 11,
       11, 11, 12, 12, 12, 12, 12, 13, 13, 13, 14, 14, 14, 14, 14, 14, 14,
       14, 14, 14, 14, 14, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15,
       15, 15, 16, 16, 16

In [17]:
torch.save(community_dict, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/StimOn/community_stimOn_dict.pt')

In [22]:
key = 'stimOn_times'

In [21]:
community_dict

{1.0: {'communities': array([ 21, 195, 196,   2,   3,  11,  12,  13,  15,  65, 127, 128, 129,
         130, 131, 132, 133, 134, 138, 139, 143, 163, 213, 236,   6,   7,
           8,  16,  18,  19,  36,  37, 146, 147, 157, 158, 165, 166, 167,
         226, 239, 240, 241, 242, 243, 255, 308, 441, 137, 140, 141, 142,
         150, 151, 155, 113, 114, 115, 177, 182,  56, 100, 109, 181,  99,
         101, 102, 108, 110, 111, 112, 169, 170, 171, 172, 173, 174, 175,
         176, 179, 184, 185, 186, 188, 225, 319, 320, 321, 435,  93,  94,
          95,  96,  97, 116, 120, 366, 385, 388, 389, 390, 391, 393, 394,
         395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 406, 407, 408,
         409, 410, 411, 412, 413, 415, 417, 418, 419, 420, 454, 455, 456,
         457, 458, 459, 460, 461, 462, 463, 464, 465, 467, 468, 469, 470,
         471, 258, 259, 260, 262, 264, 269, 273, 274, 276, 284, 297, 316,
         317, 318, 325, 328, 331, 332, 333, 335, 336, 337, 338, 340, 341,
         342, 343,

In [24]:
communities_accu_indiv = {}
key = 'stimOn_times'
for resolution in [1.0]:
    communities = community_dict[resolution]['communities']
    communities_label = community_dict[resolution]['communities_label']
    communities_acronym = community_dict[resolution]['communities_acronym']

    id_community_mapping = []
    for acronym in acronym_list:
        if acronym in communities_acronym:
            id_community_mapping.append(communities_label[np.argwhere(communities_acronym == acronym).flatten()])
        else:
            id_community_mapping.append(np.array([-1]))

    id_community_mapping = np.array(id_community_mapping)

    brain_region_index_intersect = id_community_mapping[brain_region_index].squeeze(-1)
    dropout_index = np.argwhere(brain_region_index_intersect == np.array([-1]))[:, 0]

    brain_region_index_intersect = torch.tensor(brain_region_index_intersect, dtype=torch.long)

    ind = 0
    model_dir = f'/content/drive/MyDrive/Project/BrainRegionId/Science/Model/Community/StimOn/{resolution}'

    train_args = {
        'overfitting_thres':0.60,
        'lr':5e-4,
        'norm':True,
        'temp':[True, True],
        'epochs':50,

    }

    Classifier_AnyNet = torch.load(model_dir + f'/AnyNet_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_ViT = torch.load(model_dir + f'/ViT_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)
    Classifier_RNN = torch.load(model_dir + f'/RNN_L_AllenCommunity_{key}_{ind}.pth', weights_only=False).to(device)

    subject_od_ind = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science' + f'/Model/Allen/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
    subject_od_list = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science' + f'/Model/Allen/subject_od_list_Allen_{key}{0}.pt', weights_only=False)
    train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)

    train_ind = np.setdiff1d(train_ind, dropout_index)
    valid_ind = np.setdiff1d(valid_ind, dropout_index)
    test_ind = np.setdiff1d(test_ind, dropout_index)

    acu_test = []
    community_test = []
    community_index_test = []
    model_type = []

    for community in np.unique(community_dict[resolution]['communities_label']):

        test_ind_acronym_ii = np.intersect1d(np.argwhere(np.array(brain_region_index_intersect) == community).flatten(), test_ind)

        if len(test_ind_acronym_ii) < 1:

            community_test.append(community)

            community_index_test.append(community)

            acu_test.append(np.nan)

            model_type.append(Classifier_name[Classifier_ii])

        else:
            test_indiv = test_ind_acronym_ii
            data_test_indiv = TensorDataset(brain_signal_lfp[test_indiv,:], brain_region_index_intersect[test_indiv])
            test_iter_indiv = DataLoader(data_test_indiv, batch_size=128, shuffle=True)


            Classifier_name = ['AnyNet', 'ViT', 'RNN']
            for Classifier_ii, Classifier in enumerate([Classifier_AnyNet, Classifier_ViT, Classifier_RNN]):
                Classifier.eval()
                for x_test1, y_test in test_iter_indiv:
                    if Classifier_name[Classifier_ii] == 'RNN':
                        x_test = lfp_spectro(x_test1, spectro_args, train_args)
                        y_test = y_test.to(device)
                        py_test = Classifier(x_test.to(device).squeeze(1).permute(0, 2, 1))
                        del x_test, x_test1
                        acu_test.append(accu_fun(py_test, y_test))
                        model_type.append(Classifier_name[Classifier_ii])

                    elif Classifier_name[Classifier_ii] in ['Chance', 'Linear']:
                        x_test = lfp_spectro(x_test1, spectro_args, train_args)
                        y_test = y_test.to(device)
                        py_test = Classifier(x_test.to(device).squeeze(1).flatten(start_dim=1))
                        del x_test, x_test1
                        acu_test.append(accu_fun(py_test, y_test))
                        model_type.append(Classifier_name[Classifier_ii])

                    else:
                        x_test = lfp_spectro(x_test1, spectro_args, train_args)
                        y_test = y_test.to(device)
                        py_test = Classifier(x_test.to(device))
                        del x_test, x_test1
                        acu_test.append(accu_fun(py_test, y_test))
                        model_type.append(Classifier_name[Classifier_ii])

                    community_test.append(community)
                    community_index_test.append(community)

    acu = pd.DataFrame({
        'acu_test': np.array(acu_test),
        'model_type': model_type,
        'acronym_test': community_test,
        'acronym_index_test': community_index_test,
    })

    communities_accu_indiv[resolution] = acu

In [25]:
torch.save(communities_accu_indiv, '/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/StimOn/communities_stimOn_accu_indiv.pt')

In [ ]:
from google.colab import runtime
runtime.unassign()